# transform_customers

Cleans the synthetic `customers` table into `customers_clean`. FabricOps sandbox only.

In [ ]:
from pyspark.sql import functions as F

source = spark.read.table("customers")
clean = source.select(
    F.col("customer_id"),
    F.trim("full_name").alias("full_name"),
    F.upper("country").alias("country"),
    F.col("segment"),
    F.to_date("signup_date").alias("signup_date"),
    F.col("lifetime_value").cast("double").alias("lifetime_value"),
).withColumn(
    "value_band",
    F.when(F.col("lifetime_value") < 250, "low").when(F.col("lifetime_value") < 600, "medium").otherwise("high"),
)
clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("customers_clean")
print(f"transformed {clean.count()} customers")